# SOLARNESIA: Spatial Model & GIS Analytics for National Solar Power Plant (PLTS) Distribution
**Mata Kuliah:** IMT01303501 - Komputer Grafis (3 SKS)  
**Kelompok 6:** Wesley Goeinawan (0706012410011), Stevanus Ivan Santoso (0706012410029), Nicholas Gerwin Mawardji (0706012410015)  
**Program Studi:** Informatika, Universitas Ciputra Surabaya (2026/2027)

---
### Ringkasan Alur Pemrosesan Spasial
1. **Layer 1 (2D MultiPolygons):** Batas daratan kepulauan dan batas administrasi kabupaten/provinsi (Choropleth rata-rata potensi PVOUT nasional).
2. **Layer 2 (0D Points & Buffers):** Titik observasi Global Solar Atlas darat murni (3.802 titik) setelah *Spatial Clip* mengeliminasi 1.160 titik laut.
3. **5 km Hotspot Clustering:** Pengelompokan titik kandidat berdekatan (≤ 5 km) menjadi sentra hotspot terintegrasi (DBSCAN).
4. **Operasi 1 (Buffer Metrik):** Proyeksi EPSG:3857 dengan koreksi $\cos(\text{lat})$, pembentukan *footprint* 3 km `buffer(3000)` dan koridor transmisi 25 km `buffer(25000)` (standar PLN 20 kV).
5. **Operasi 2 (Spatial Join & Overlay):** Menghitung keterjangkauan 3.289 permukiman darat serta mengekstrak daftar kecamatan/kabupaten yang tercakup secara interaktif.


In [1]:
# Cell 0: Setup & Library Imports
import os
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from shapely.geometry import Point, Polygon
from sklearn.cluster import DBSCAN
import folium



In [2]:
# Cell 1: Ingestion & GeoDataFrame Construction
DATA_DIR = "data"

# 1. Load Solar Observation Points (Global Solar Atlas)
df_raw = pd.read_csv(os.path.join(DATA_DIR, "indonesia_solar_data.csv"))
raw_pts = gpd.GeoDataFrame(
    df_raw,
    geometry=gpd.points_from_xy(df_raw["longitude"], df_raw["latitude"]),
    crs="EPSG:4326"
)

# 2. Load Administrative Boundaries (Provinces & Kabupaten)
gdf_prov = gpd.read_file(os.path.join(DATA_DIR, "indonesia_provinces.geojson"))
gdf_kab = gpd.read_file(os.path.join(DATA_DIR, "indonesia_kabupaten.geojson"))

assert len(raw_pts) == 4962, f"Expected 4962 points, got {len(raw_pts)}"
assert raw_pts.crs.to_epsg() == 4326


In [3]:
# Cell 2: Spatial Clip / Land-Masking (Op 0)
# Mengeliminasi titik-titik observasi yang jatuh di perairan laut menggunakan poligon daratan kabupaten
land_sj = gpd.sjoin(raw_pts, gdf_kab[["shapeName", "geometry"]], how="inner", predicate="intersects")
land_pts = raw_pts.loc[land_sj.index.unique()].copy()

ocean_pts_count = len(raw_pts) - len(land_pts)
assert len(land_pts) == 3802, f"Expected 3802 points, got {len(land_pts)}"
assert ocean_pts_count == 1160


In [4]:
# Cell 3: Site Selection & Role Classification (Layer 2)
# Ambang batas: PVOUT >= Persentil ke-75 (1.430 kWh/kWp) dan Suhu Rata-rata <= 27,5 °C
p75_pvout = 1430.0
max_temp = 27.5

is_plts = (land_pts["pvout"] >= p75_pvout) & (land_pts["temp"] <= max_temp)
plts_candidates = land_pts[is_plts].copy()
demand_points = land_pts[~is_plts].copy()

plts_candidates["role"] = "PLTS_CANDIDATE"
demand_points["role"] = "DEMAND"

assert len(plts_candidates) == 513, f"Expected 513 PLTS candidates, got {len(plts_candidates)}"
assert len(demand_points) == 3289, f"Expected 3289 demand points, got {len(demand_points)}"


In [5]:
# Cell 4: 5 km Proximity Hotspot Clustering (DBSCAN)
# Titik-titik tapak PLTS dalam radius <= 5 km dikelompokkan ke dalam satu sentra klaster energi
plts_3857 = plts_candidates.to_crs("EPSG:3857")
coords_m = np.column_stack([plts_3857.geometry.x, plts_3857.geometry.y])

db_5k = DBSCAN(eps=5000, min_samples=1, metric="euclidean").fit(coords_m)
plts_candidates["cluster_id"] = db_5k.labels_
plts_3857["cluster_id"] = db_5k.labels_

plts_candidates["hotspot_id"] = [f"HOTSPOT-{cid:03d}" for cid in db_5k.labels_]
plts_3857["hotspot_id"] = plts_candidates["hotspot_id"]

unique_hotspots = plts_candidates["hotspot_id"].nunique()
multi_site_hotspots = plts_candidates.groupby("hotspot_id").filter(lambda g: len(g) > 1)["hotspot_id"].nunique()

assert unique_hotspots == 480, f"Expected 480 hotspots, got {unique_hotspots}"
assert multi_site_hotspots == 31


In [6]:
# Cell 5: Transformasi CRS & Pembuatan Buffer Metrik (Op 1)
# Transformasi ke EPSG:3857 dengan koreksi distorsi Mercator: R_adj = R / cos(latitude)
demand_3857 = demand_points.to_crs("EPSG:3857")

lats_plts = plts_candidates["latitude"].values
rad_adj_3k = 3000.0 / np.cos(np.radians(lats_plts))
rad_adj_25k = 25000.0 / np.cos(np.radians(lats_plts))

# Buffer 3 km (Footprint Lahan) dan Buffer 25 km (Standar Distribusi 20 kV PLN)
plts_3857["footprint_3km"] = [geom.buffer(r) for geom, r in zip(plts_3857.geometry, rad_adj_3k)]
plts_3857["buffer_25km"] = [geom.buffer(r) for geom, r in zip(plts_3857.geometry, rad_adj_25k)]

# Buffer Sensitivitas 10 km dan 40 km
rad_adj_10k = 10000.0 / np.cos(np.radians(lats_plts))
rad_adj_40k = 40000.0 / np.cos(np.radians(lats_plts))
plts_3857["buffer_10km"] = [geom.buffer(r) for geom, r in zip(plts_3857.geometry, rad_adj_10k)]
plts_3857["buffer_40km"] = [geom.buffer(r) for geom, r in zip(plts_3857.geometry, rad_adj_40k)]

# Poligon Union Koridor Distribusi
union_25km = gpd.GeoSeries(plts_3857["buffer_25km"], crs="EPSG:3857").union_all()
union_10km = gpd.GeoSeries(plts_3857["buffer_10km"], crs="EPSG:3857").union_all()
union_40km = gpd.GeoSeries(plts_3857["buffer_40km"], crs="EPSG:3857").union_all()


In [7]:
# Cell 6: Analisis Keterjangkauan Permukiman Spasial (Op 2)
# Titik demand diuji posisinya terhadap poligon buffer distribusi menggunakan predikat intersects / within
demand_3857["covered_25km"] = demand_3857.geometry.within(union_25km)
demand_3857["covered_10km"] = demand_3857.geometry.within(union_10km)
demand_3857["covered_40km"] = demand_3857.geometry.within(union_40km)

reached_25 = int(demand_3857["covered_25km"].sum())
unreached_25 = len(demand_3857) - reached_25
pct_25 = (reached_25 / len(demand_3857)) * 100

reached_10 = int(demand_3857["covered_10km"].sum())
pct_10 = (reached_10 / len(demand_3857)) * 100

reached_40 = int(demand_3857["covered_40km"].sum())
pct_40 = (reached_40 / len(demand_3857)) * 100

assert reached_25 == 325, f"Expected 325, got {reached_25}"
assert unreached_25 == 2964, f"Expected 2964, got {unreached_25}"
assert round(pct_25, 1) == 9.9
assert reached_10 == 49
assert reached_40 == 605


In [8]:
# Cell 7: Ekstraksi Cakupan Wilayah Administratif per Titik / Hotspot
# Melakukan spatial join antara buffer 25 km setiap titik PLTS dengan poligon batas kabupaten
kab_3857 = gdf_kab.to_crs("EPSG:3857")

covered_districts = []
for idx, row in plts_3857.iterrows():
    buf_poly = row["buffer_25km"]
    intersecting_kabs = kab_3857[kab_3857.geometry.intersects(buf_poly)]["shapeName"].unique()
    covered_districts.append(", ".join(sorted(intersecting_kabs)))

plts_candidates["covered_areas_25km"] = covered_districts
plts_3857["covered_areas_25km"] = covered_districts

# Gabungan seluruh kabupaten yang terlayani
all_covered_kabs = set()
for area_str in covered_districts:
    if area_str:
        all_covered_kabs.update([k.strip() for k in area_str.split(",")])

assert len(all_covered_kabs) == 294


In [9]:
# Cell 8: Rangkuman Metrik Tabel 1
summary_table1 = pd.DataFrame([
    {
        "Radius Buffer": "10 km (Sensitivitas)",
        "Permukiman Terjangkau": reached_10,
        "Persentase": f"{pct_10:.1f}%".replace(".", ",")
    },
    {
        "Radius Buffer": "25 km (Standar Resmi PLN)",
        "Permukiman Terjangkau": reached_25,
        "Persentase": f"{pct_25:.1f}%".replace(".", ",")
    },
    {
        "Radius Buffer": "40 km (Sensitivitas)",
        "Permukiman Terjangkau": reached_40,
        "Persentase": f"{pct_40:.1f}%".replace(".", ",")
    }
])

summary_table1


,Radius Buffer,Permukiman Terjangkau,Persentase
0,10 km (Sensitivitas),49,"1,5%"
1,25 km (Standar Resmi PLN),325,"9,9%"
2,40 km (Sensitivitas),605,"18,4%"


In [10]:
# Cell 9: Peta Spasial Interaktif & Inspeksi Klik (Folium)
# Mengintegrasikan Layer 1 (Choropleth Poligon Provinsi) dan Layer 2 (Titik PLTS, Koridor Buffer 25 km, serta Titik Permukiman)
# Center map on Indonesia (Bali & Java focal area)
center_lat, center_lon = -8.40, 115.20
m = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=9,
    tiles="OpenStreetMap",
    control_scale=True
)

# 1. Layer 1: Koroplet Potensi Surya Provinsi (34 Provinsi - Toggleable)
prov_means = gpd.sjoin(land_pts, gdf_prov, how='inner', predicate='intersects').groupby('state')['pvout'].mean()
gdf_prov['mean_pvout'] = gdf_prov['state'].map(prov_means).fillna(land_pts['pvout'].mean()).round(1)

prov_layer = folium.FeatureGroup(name="Layer 1: Koroplet Potensi Surya Provinsi (34 Provinsi)", show=False)

def prov_style(feature):
    val = feature['properties'].get('mean_pvout', 1400)
    if val >= 1500:
        c = '#dc2626'
    elif val >= 1420:
        c = '#ea580c'
    elif val >= 1350:
        c = '#f59e0b'
    else:
        c = '#fde047'
    return {'fillColor': c, 'color': '#334155', 'weight': 1.2, 'fillOpacity': 0.35}

folium.GeoJson(
    gdf_prov,
    style_function=prov_style,
    tooltip=folium.GeoJsonTooltip(fields=['state', 'mean_pvout'], aliases=['Provinsi:', 'Rata-rata PVOUT (kWh/kWp):'])
).add_to(prov_layer)
prov_layer.add_to(m)

# 2. Layer 2: Koridor Distribusi 25 km (PLN 20 kV)
buffer_layer = folium.FeatureGroup(name="Koridor Distribusi 25 km (PLN 20 kV)")
for idx, row in plts_candidates.iterrows():
    buf_circle = folium.Circle(
        location=[row['latitude'], row['longitude']],
        radius=25000,
        color="#38bdf8",
        weight=1.5,
        fill=True,
        fill_color="#38bdf8",
        fill_opacity=0.15,
        tooltip=f"<b>Hotspot:</b> {row['hotspot_id']}<br><b>Radius:</b> 25 km"
    )
    buf_circle.add_to(buffer_layer)
buffer_layer.add_to(m)

demand_4326 = demand_3857.to_crs("EPSG:4326")

# 3. Layer 2: Permukiman Terjangkau 25 km (Emerald Dots)
demand_layer = folium.FeatureGroup(name="Permukiman Terjangkau 25 km (325 Titik)")
for idx, row in demand_4326[demand_4326['covered_25km']].iterrows():
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=3.5,
        color="#10b981",
        fill=True,
        fill_color="#10b981",
        fill_opacity=0.85,
        tooltip=f"Demand ID: {idx} | Terjangkau"
    ).add_to(demand_layer)
demand_layer.add_to(m)

# 4. Layer 2: Permukiman Belum Terjangkau (Slate Dots - Toggleable)
unreached_layer = folium.FeatureGroup(name="Permukiman Belum Terjangkau (2.964 Titik)", show=False)
for idx, row in demand_4326[~demand_4326['covered_25km']].iterrows():
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=2.5,
        color="#64748b",
        fill=True,
        fill_color="#64748b",
        fill_opacity=0.35,
        weight=0.5,
        tooltip=f"Demand ID: {idx} | Belum Terjangkau (>25km)"
    ).add_to(unreached_layer)
unreached_layer.add_to(m)

# 5. Layer 2: Kandidat PLTS & Hotspot Sentra dengan Click Popup
plts_layer = folium.FeatureGroup(name="Kandidat PLTS & Hotspot Sentra (513 Titik)")
for idx, row in plts_candidates.iterrows():
    popup_html = f"""
    <div style='font-family: Arial, sans-serif; font-size: 12px; width: 260px; line-height: 1.5;'>
        <b style='color: #d97706; font-size: 13px;'>☀️ {row['hotspot_id']}</b><br>
        <b>Koordinat:</b> {row['latitude']:.4f}° LS, {row['longitude']:.4f}° BT<br>
        <b>Potensi PVOUT:</b> <span style='color: #059669; font-weight: bold;'>{row['pvout']:.1f} kWh/kWp</span><br>
        <b>Suhu Rata-rata:</b> {row['temp']:.1f} °C<br>
        <hr style='margin: 6px 0; border: 0; border-top: 1px solid #e5e7eb;'>
        <b>Wilayah Terjangkau (Radius 25 km):</b><br>
        <span style='color: #1e3a8a; font-size: 11px;'>{row['covered_areas_25km']}</span>
    </div>
    """
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=6,
        color="#ffffff",
        weight=1.5,
        fill=True,
        fill_color="#f59e0b",
        fill_opacity=0.95,
        popup=folium.Popup(popup_html, max_width=300),
        tooltip=f"<b>Klik untuk Inspeksi:</b> {row['hotspot_id']} (PVOUT: {row['pvout']:.0f})"
    ).add_to(plts_layer)
plts_layer.add_to(m)

folium.LayerControl(collapsed=False).add_to(m)
m
